# Data Profiling: Green - NYC TLC

Este notebook realiza el perfil exploratorio de los datos de los taxis verdes (Green Cab) del proyecto NYC TLC.
Los taxis verdes (Boro Taxis) operan principalmente en los barrios de Nueva York fuera de Manhattan,
como el Bronx, Brooklyn, Queens y Staten Island, asi como en el norte de Manhattan.
El objetivo es comprender la estructura, distribucion y calidad basica de los datos en crudo.


In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work')
import os
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import seaborn as sns
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import *
from src.spark_utils import get_spark
from src.paths import RAW

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

VEHICLE = 'green'
RAW_PATH = str(RAW[VEHICLE])
print(f'Raw path: {RAW_PATH}')


## 1. Inicializacion de Spark


In [ ]:
spark = get_spark(app_name=f'TLC_Profiling_{VEHICLE.upper()}')

# Leer todos los archivos parquet del directorio raw
df = spark.read.parquet(RAW_PATH)

print(f'Total registros: {df.count():,}')
print(f'Total columnas: {len(df.columns)}')
df.printSchema()


## 2. Perfil General: Estadisticas Descriptivas


In [ ]:
# Estadisticas descriptivas generales
stats = df.describe().toPandas()
print('Estadisticas descriptivas:')
try:
    display(stats)
except NameError:
    print(stats.to_string())

# Conteo de nulos por columna
total_registros = df.count()
null_counts = df.select(
    [F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df.columns]
).toPandas().T
null_counts.columns = ['nulos']
null_counts['porcentaje'] = (null_counts['nulos'] / total_registros * 100).round(2)

print('\nValores nulos por columna:')
nulos_presentes = null_counts[null_counts['nulos'] > 0].sort_values('porcentaje', ascending=False)
try:
    display(nulos_presentes)
except NameError:
    print(nulos_presentes.to_string())


## 3. Distribucion de Variables Numericas


In [ ]:
# Muestra del 5% para rendimiento en graficos
# Green taxi usa lpep_pickup_datetime (en lugar de tpep_*)
columnas_numericas = ['trip_distance', 'fare_amount', 'total_amount', 'tip_amount', 'passenger_count']
df_sample = df.select(columnas_numericas).sample(fraction=0.05, seed=42).toPandas()

# Filtrar valores extremos para mejor visualizacion
df_sample = df_sample[
    (df_sample['trip_distance'] >= 0) & (df_sample['trip_distance'] <= 100) &
    (df_sample['fare_amount'] >= 0) & (df_sample['fare_amount'] <= 300) &
    (df_sample['total_amount'] >= 0) & (df_sample['total_amount'] <= 300) &
    (df_sample['tip_amount'] >= 0) & (df_sample['tip_amount'] <= 100) &
    (df_sample['passenger_count'] >= 1) & (df_sample['passenger_count'] <= 6)
]

fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

titulos = {
    'trip_distance': 'Distancia del viaje (millas)',
    'fare_amount': 'Tarifa base (USD)',
    'total_amount': 'Monto total (USD)',
    'tip_amount': 'Propina (USD)',
    'passenger_count': 'Cantidad de pasajeros'
}

posiciones = [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1)]

for idx, col in enumerate(columnas_numericas):
    fila, columna = posiciones[idx]
    ax = fig.add_subplot(gs[fila, columna])
    datos_col = df_sample[col].dropna()
    ax.hist(datos_col, bins=40, color='seagreen', edgecolor='white', alpha=0.85)
    ax.set_title(titulos[col], fontsize=11, fontweight='bold')
    ax.set_xlabel('Valor', fontsize=9)
    ax.set_ylabel('Frecuencia', fontsize=9)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    media = datos_col.mean()
    ax.axvline(media, color='red', linestyle='--', linewidth=1.2, label=f'Media: {media:.2f}')
    ax.legend(fontsize=8)

# Ocultar subplot vacio
ax_vacio = fig.add_subplot(gs[1, 2])
ax_vacio.set_visible(False)

fig.suptitle('Distribucion de Variables Numericas - Green Taxi', fontsize=14, fontweight='bold', y=1.01)
plt.savefig('dist_numericas_green.png', bbox_inches='tight', dpi=120)
plt.show()
print('Grafico guardado: dist_numericas_green.png')


## 4. Distribucion Temporal: Viajes por Ano y Mes


In [ ]:
# Green taxi usa lpep_pickup_datetime (no tpep_pickup_datetime)
df_temporal = df.withColumn('anio', F.year('lpep_pickup_datetime')) \
                .withColumn('mes', F.month('lpep_pickup_datetime')) \
                .withColumn('anio_mes', F.date_format('lpep_pickup_datetime', 'yyyy-MM'))

# Agrupar y contar viajes por anio-mes
conteo_temporal = df_temporal.groupBy('anio_mes') \
    .agg(F.count('*').alias('viajes')) \
    .filter(F.col('anio_mes').isNotNull()) \
    .orderBy('anio_mes') \
    .toPandas()

# Filtrar solo anos razonables
conteo_temporal = conteo_temporal[conteo_temporal['anio_mes'].str[:4].isin(
    [str(y) for y in range(2019, 2026)]
)]

fig, ax = plt.subplots(figsize=(16, 5))
sns.barplot(data=conteo_temporal, x='anio_mes', y='viajes', color='seagreen', ax=ax)
ax.set_title('Viajes por Ano y Mes - Green Taxi', fontsize=13, fontweight='bold')
ax.set_xlabel('Periodo (Ano-Mes)', fontsize=10)
ax.set_ylabel('Cantidad de Viajes', fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Rotar etiquetas del eje x
n_etiquetas = len(conteo_temporal)
paso = max(1, n_etiquetas // 24)
etiquetas = [label if i % paso == 0 else '' for i, label in enumerate(conteo_temporal['anio_mes'])]
ax.set_xticklabels(etiquetas, rotation=45, ha='right', fontsize=8)

plt.tight_layout()
plt.savefig('temporal_green.png', bbox_inches='tight', dpi=120)
plt.show()
print('Grafico guardado: temporal_green.png')


## 5. Top 10 Zonas de Mayor Demanda (Pickup)


In [ ]:
# Top 10 zonas de recogida por cantidad de viajes
top_zonas = df.groupBy('PULocationID') \
    .agg(F.count('*').alias('viajes')) \
    .filter(F.col('PULocationID').isNotNull()) \
    .orderBy(F.desc('viajes')) \
    .limit(10) \
    .toPandas()

top_zonas['PULocationID'] = top_zonas['PULocationID'].astype(str)
top_zonas = top_zonas.sort_values('viajes', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colores = sns.color_palette('Greens_r', n_colors=len(top_zonas))
bars = ax.barh(top_zonas['PULocationID'], top_zonas['viajes'], color=colores)

# Etiquetas de valor al final de cada barra
for bar_elem in bars:
    ancho = bar_elem.get_width()
    ax.text(ancho * 1.01, bar_elem.get_y() + bar_elem.get_height() / 2,
            f'{ancho:,.0f}', va='center', fontsize=9)

ax.set_title('Top 10 Zonas de Mayor Demanda (Pickup) - Green Taxi', fontsize=13, fontweight='bold')
ax.set_xlabel('Cantidad de Viajes', fontsize=10)
ax.set_ylabel('ID de Zona (PULocationID)', fontsize=10)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig('top_zonas_green.png', bbox_inches='tight', dpi=120)
plt.show()
print('Grafico guardado: top_zonas_green.png')


## 6. Distribucion de Metodos de Pago


In [ ]:
# Mapeo de codigos de pago a etiquetas legibles
mapa_pago = {
    1: 'Tarjeta',
    2: 'Efectivo',
    3: 'Sin cargo',
    4: 'Disputa',
    5: 'Desconocido',
    6: 'Anulado'
}

conteo_pago = df.groupBy('payment_type') \
    .agg(F.count('*').alias('viajes')) \
    .filter(F.col('payment_type').isNotNull()) \
    .orderBy('payment_type') \
    .toPandas()

conteo_pago['etiqueta'] = conteo_pago['payment_type'].map(
    lambda x: mapa_pago.get(int(x), f'Tipo {int(x)}') if x is not None else 'Desconocido'
)

colores_pago = sns.color_palette('Set2', n_colors=len(conteo_pago))

fig, ax = plt.subplots(figsize=(8, 8))
wedges, textos, autotextos = ax.pie(
    conteo_pago['viajes'],
    labels=conteo_pago['etiqueta'],
    autopct='%1.1f%%',
    colors=colores_pago,
    startangle=140,
    pctdistance=0.82,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
for texto in autotextos:
    texto.set_fontsize(10)
for texto in textos:
    texto.set_fontsize(11)

ax.set_title('Distribucion de Metodos de Pago - Green Taxi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('pago_green.png', bbox_inches='tight', dpi=120)
plt.show()
print('Grafico guardado: pago_green.png')


## 7. Correlacion entre Variables Numericas


In [ ]:
# Columnas numericas para la matriz de correlacion
# Green taxi incluye trip_type (1=Calle, 2=Despacho)
cols_corr = ['trip_distance', 'fare_amount', 'tip_amount', 'total_amount',
             'tolls_amount', 'passenger_count']

# Calcular correlacion usando Spark
matriz_corr = pd.DataFrame(index=cols_corr, columns=cols_corr, dtype=float)

for col_a in cols_corr:
    for col_b in cols_corr:
        try:
            val = df.stat.corr(col_a, col_b)
        except Exception:
            val = float('nan')
        matriz_corr.loc[col_a, col_b] = round(val, 4)

# Etiquetas en espanol para el grafico
etiquetas_corr = {
    'trip_distance': 'Distancia',
    'fare_amount': 'Tarifa',
    'tip_amount': 'Propina',
    'total_amount': 'Total',
    'tolls_amount': 'Peajes',
    'passenger_count': 'Pasajeros'
}
etiquetas = [etiquetas_corr[c] for c in cols_corr]

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    matriz_corr.astype(float),
    annot=True,
    fmt='.3f',
    cmap='coolwarm',
    center=0,
    vmin=-1,
    vmax=1,
    xticklabels=etiquetas,
    yticklabels=etiquetas,
    ax=ax,
    linewidths=0.5
)
ax.set_title('Correlacion entre Variables Numericas - Green Taxi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlacion_green.png', bbox_inches='tight', dpi=120)
plt.show()
print('Grafico guardado: correlacion_green.png')


## 8. Distribucion Horaria de Viajes


In [ ]:
# Green taxi usa lpep_pickup_datetime
conteo_hora = df.withColumn('hora', F.hour('lpep_pickup_datetime')) \
    .groupBy('hora') \
    .agg(F.count('*').alias('viajes')) \
    .filter(F.col('hora').isNotNull()) \
    .orderBy('hora') \
    .toPandas()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(conteo_hora['hora'], conteo_hora['viajes'],
        marker='o', linewidth=2.2, color='seagreen', markersize=6)
ax.fill_between(conteo_hora['hora'], conteo_hora['viajes'], alpha=0.15, color='seagreen')

ax.set_title('Distribucion Horaria de Viajes - Green Taxi', fontsize=13, fontweight='bold')
ax.set_xlabel('Hora del dia (0-23)', fontsize=10)
ax.set_ylabel('Cantidad de Viajes', fontsize=10)
ax.set_xticks(range(0, 24))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(True, linestyle='--', alpha=0.6)

# Marcar hora pico
hora_pico = conteo_hora.loc[conteo_hora['viajes'].idxmax(), 'hora']
viajes_pico = conteo_hora['viajes'].max()
ax.annotate(f'Hora pico: {hora_pico}h',
            xy=(hora_pico, viajes_pico),
            xytext=(hora_pico + 1.5, viajes_pico * 0.95),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=10, color='red')

plt.tight_layout()
plt.savefig('horario_green.png', bbox_inches='tight', dpi=120)
plt.show()
print('Grafico guardado: horario_green.png')


In [ ]:
spark.stop()
print('Profiling completado.')
